# Data Exploration and Preprocessing

In [1]:
import pandas as pd
import duckdb
from pathlib import Path

## Download Instructions

If raw data is not saved as `data/raw/Books.jsonl.gz` and `data/raw/meta_Books.jsonl.gz`, follow the instructions in `README.md`

## Create Parquet Files If Needed

In [ ]:
# Full files

if not Path("../data/raw/Books.parquet").is_file():
    try:
        duckdb.query("COPY (SELECT * FROM read_json_auto('../data/raw/Books.jsonl.gz')) TO '../data/raw/Books.parquet' (FORMAT PARQUET)")
    except:
        print("Parquet file not created.")
        print("Do you have the file '../data/raw/Books.jsonl.gz'?")
if not Path("../data/raw/meta_Books.parquet").is_file():
    try:
        duckdb.query("COPY (SELECT * FROM read_json_auto('../data/raw/meta_Books.jsonl.gz', sample_size = 1000000)) TO '../data/raw/meta_Books.parquet' (FORMAT PARQUET)")
    except:
        print("Parquet file not created.")
        print("Do you have the file '../data/raw/met_Books.jsonl.gz'?")

In [ ]:
# Top rated books

n_books = 200

con = duckdb.connect()

# get top books
con.execute(f"""
CREATE TEMP TABLE top_books AS
SELECT *
FROM read_json_auto("../data/raw/meta_Books.jsonl.gz", sample_size = 1000000)
WHERE rating_number > 10
ORDER BY average_rating DESC
LIMIT {n_books};
""")

# save top book meta data
con.execute("""
COPY top_books
TO "../data/raw/top_meta_Books.parquet"
(FORMAT PARQUET)
""")

# get and save top book reviews
con.execute(f"""
COPY (
    SELECT *
    FROM read_json_auto("../data/raw/Books.jsonl.gz")
    WHERE asin IN (SELECT parent_asin FROM top_books)
) TO "../data/raw/top_Books.parquet";
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Dataset Overview

In [3]:
# fields in reviews file
duckdb.query("""
       DESCRIBE SELECT * FROM "../data/raw/Books.parquet";
""")

┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │                                                  column_type                                                  │  null   │   key   │ default │  extra  │
│      varchar      │                                                    varchar                                                    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ rating            │ DOUBLE                                                                                                        │ YES     │ NULL    │ NULL    │ NULL    │
│ title             │ VARCHAR                                                                                                     

In [4]:
# number of reviews
duckdb.query("""
       SELECT COUNT(*) FROM "../data/raw/Books.parquet";
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     29475453 │
└──────────────┘

In [5]:
# fields in meta books files
duckdb.query("""
       DESCRIBE SELECT * FROM "../data/raw/meta_Books.parquet";
""")

┌─────────────────┬───────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │                                column_type                                │  null   │   key   │ default │  extra  │
│     varchar     │                                  varchar                                  │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼───────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ main_category   │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ title           │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ subtitle        │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ author          │ STRUCT(avatar VARCHAR, "name

In [6]:
# number of books
duckdb.query("""
       SELECT COUNT(*) FROM "../data/raw/meta_Books.parquet";
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      4448181 │
└──────────────┘

## Sample Records

### Review Records

In [17]:
books = pd.read_parquet("../data/raw/top_Books.parquet",)
books.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Fun winter read,My 6 year olds love this book and can read it ...,[],1416925376,1416925376,AE5IDXXADTXDLZWICJDTPIADYDIA,1509818114237,0,True
1,5.0,Love the repetition,Love how this repeats and repeats and builds o...,[],B00CB946QC,B00CB946QC,AFW6M7CWMABZPQSO7HNRGAYVPEAA,1628431957037,1,True
2,5.0,Rusty's life from puppy mill to a life of love.,I really loved this book. I bought one as a gi...,[],0692879412,0692879412,AFC7WI3L64JDBMTPAD5AARFUOPTA,1524687698029,0,True
3,5.0,A Must Have!!!,My little cousin was so inspired by this book!,[],1735063509,1735063509,AF5UMCW55TK35V7YGL7Z66AX3XWQ,1608248859933,0,True
4,5.0,A new look at an old prejudice,Sharon Jayne's book examines how God truly rel...,[],0736930469,0736930469,AEJB2EVKHD6VOX7XAHXDDYNWRVUQ,1390218520000,1,True


In [18]:
books.tail()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
912,5.0,breathtaking!!,"I couldn't put this book down, it was spellbin...",[],1936919427,1936919427,AGODD7YKSUCIZZHXQ7JJXYV7WXGQ,1539813557855,1,False
913,5.0,The Real Deal!,I loved THE REAL FARMER AND THE DELL. What a c...,[],1946101885,1946101885,AEXHUSBLOZSWUY7QH3H6J7TQRYSQ,1661991691185,1,False
914,5.0,Rex,Another bases-loaded home run. I couldn't put...,[],172973734X,172973734X,AHDZD4YYAZ5U3B3KQYFFLYARYQCA,1551286041282,0,True
915,5.0,Absolutely amazing.,Such an amazing book and my little nephew love...,[],1735501905,1735501905,AGVFIQPJU6ENAY4DQEPBXFZAEENA,1624559131887,1,True
916,5.0,A Perfect Read Aloud!,This charming story of a very worried family i...,[],1945058064,1945058064,AGE4OIGSO6TTMTITCJCTHJ3OPF6A,1459842480000,1,True


### Meta Book Records

In [19]:
meta_books = pd.read_parquet("../data/raw/top_meta_Books.parquet")
meta_books.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Gauguin: The Quest for Paradise,"Paperback – Bargain Price, March 30, 1992",{'avatar': 'https://m.media-amazon.com/images/...,5.0,11,[Following the life and artistic career of one...,[],b'25.72',[{'large': 'https://m.media-amazon.com/images/...,[],Francoise Cachin (Author),"[Books, Arts & Photography, History & Criticism]","[(Publisher, b'""Harry N. Abrams (March 30, 199...",B002QGSVNY,None
1,Books,Millennials in Wonderland: Coaching Grads at t...,"Paperback – August 16, 2017",None,5.0,13,"[A successful, repeatable process for helping ...","[Review, ""Millennials in Wonderland, is a trea...",b'12.0',[{'large': 'https://m.media-amazon.com/images/...,[],"Wendy Schuman (Author), Kenneth Schuman (Author)","[Books, Business & Money, Job Hunting & Careers]","[(Publisher, b'""Grandview (August 16, 2017)""')...",0692885226,None
2,Books,Stand Watch,"Paperback – Large Print, December 17, 2022",{'avatar': 'https://m.media-amazon.com/images/...,5.0,12,[A journey of faith can be scary when faced wi...,[],b'9.95',[{'large': 'https://m.media-amazon.com/images/...,[],"Natalie Booth (Author, Illustrator), Ally Sta...","[Books, Children's Books, Literature & Fiction]","[(Publisher, b'""Natalie Booth (December 17, 20...",B0BQGJ6YJ4,None
3,Books,Pumpkin Oh Pumpkin,"Paperback – October 10, 2022",{'avatar': 'https://m.media-amazon.com/images/...,5.0,16,[Who doesn’t love to visit a pumpkin patch at ...,[],b'11.99',[{'large': 'https://m.media-amazon.com/images/...,[],Farrah Fryar (Author),"[Books, Children's Books, Growing Up & Facts o...","[(Publisher, b'""House of Inspire (October 10, ...",1957439009,None
4,Books,Broken Vessel Restored: How to Overcome Depres...,"Paperback – June 24, 2014",{'avatar': 'https://m.media-amazon.com/images/...,5.0,21,[It's a well-documented fact that in the past ...,[],b'13.51',[{'large': 'https://m.media-amazon.com/images/...,[],Wanda J. Cooper (Author),"[Books, Christian Books & Bibles, Christian Li...","[(Publisher, b'""Outskirts Press; Illustrated e...",1478733519,None


In [20]:
meta_books.tail()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
195,Books,Korean Genealogy Guide: A resource to help Eng...,"Paperback – July 25, 2012",None,5.0,12,[The Korean Genealogy Guide helps English spea...,"[About the Author, Jason Howard grew up in Sou...",b'10.0',[{'large': 'https://m.media-amazon.com/images/...,[],Jason Howard (Author),"[Books, Reference, Genealogy]","[(Publisher, b'""CreateSpace Independent Publis...",1475050151,None
196,Books,Scream Queen,"Paperback – August 11, 2021",{'avatar': 'https://m.media-amazon.com/images/...,5.0,12,"[At 23, Vera Horowitz's life is going nowhere ...",[],b'6.26',[{'large': 'https://m.media-amazon.com/images/...,[],Wray Cotterill (Author),"[Books, Humor & Entertainment, Humor]","[(Publisher, b'""Powerhouse Publishing (AUS) (A...",0578969475,None
197,Books,AI Bullseye Tactics For Non-Technical Business...,"Paperback – January 17, 2023",{'avatar': 'https://m.media-amazon.com/images/...,5.0,20,[Get real-world business results with AI—minus...,"[About the Author, Thomas Gilbertson is a seas...",b'27.99',[{'large': 'https://m.media-amazon.com/images/...,[],Thomas Gilbertson (Author),"[Books, Computers & Technology, Computer Science]","[(Publisher, b'""Grape Publishing (January 17, ...",B0BPNSFVS7,None
198,Books,Homeward: Personal Stories on the Search for B...,"Paperback – October 13, 2022",{'avatar': 'https://m.media-amazon.com/images/...,5.0,19,[Belongingness is vital to the human experienc...,[],b'16.95',[{'large': 'https://m.media-amazon.com/images/...,[],Emma Fulenwider (Author),"[Books, Literature & Fiction, Essays & Corresp...","[(Publisher, b'""Birren Center, The (October 13...",1737929627,None
199,Books,The Welsh Academy Encyclopaedia of Wales,"Hardcover – April 17, 2008",{'avatar': 'https://m.media-amazon.com/images/...,5.0,22,"[What do the Gresford Bells, Anthony Hopkins, ...","[About the Author, John Davies, is the author ...",b'98.89',[{'large': 'https://m.media-amazon.com/images/...,[],"John Davies (Editor), Nigel Jenkins (Editor),...","[Books, Reference, Encyclopedias & Subject Gui...","[(Publisher, b'""University of Wales Press; 1st...",070831953X,None


## Selection of Fields and Justification

From the review data, we will keep only asin, text, and rating. We think this in

Which fields we are keeping and removing

- Book reviews
  - KEEP
    - rating: this is helpful information to display when looking up books
    - text: this is helpful information to display when looking up books
    - asin: to connect the review to the book data
  - REMOVE
    - title: this is already in meta data
    - images: we don't be displaying images in our simple search app
    - user_id: this isn't relevant information for our search app
    - timestamp: this isn't relevant information for our search app
    - helpful_vote: this could be helpful information to use in the future but we are just keeping it simple for now
    - verified_purchase: this could be helpful information to use in the future but we are just keeping it simple for now
- Book meta data
  - KEEP
    - title: key information
    - author: keep author name but remove image and bio, the image we won't be using in this simple app and the bio might get confused with book info in dense encodings (searching for a book about Ireland might return books that are not from Ireland but are by an Irish writer)
    - average_rating: helpful information to view
    - rating_number: needed to context for average rating
    - features: this is the feature that gives the best book description, one of the most important features for searching
    - price: helpful information to view
    - parent_asin: needed to find the relevant reviews
    - categories: give genre information, this is also helpful for search
  - REMOVE
    - main_category: inconsistent information, generally says the book format which isn't important for this book search app
    - subtitle: inconsistent information, also information on format
    - description: mix of reviews and description info, not very consistent or of high quality
    - images: not needed for this search app
    - video: not needed for this search app
    - store: inconsistent data and not needed for the search app
    - details: publisher information, not needed for this search app
    - bought_together: amazon purchase information, not needed for this search app


## Text Preprocessing Decisions

## Preprocess and Save Data

In [22]:
# Save only relevant book review features
con = duckdb.connect()
con.execute("""
    COPY (
        SELECT text, rating, asin
        FROM "../data/raw/top_Books.parquet"
    ) TO "../data/processed/top_Books_processed.parquet"
    (FORMAT PARQUET)
    """
)

# Relevant meta data with some cleaning
con.execute("""
    COPY (
        SELECT 
            title,
            COALESCE(author.name, '') AS author,
            array_to_string(features, ', ') AS features,
            array_to_string(categories, ', ') AS categories,
            average_rating,
            rating_number, 
            price, 
            parent_asin
        FROM "../data/raw/top_meta_Books.parquet"
    ) TO "../data/processed/top_meta_Books_processed.parquet"
    """
)

In [ ]:
check_reviews = con.execute("""
    SELECT *
    FROM "../data/processed/Books_processed.parquet"
    LIMIT 10
    """
)

display(check_reviews.df())

,text,rating,asin
0,It is definitely not a watercolor book. The p...,1.0,B09BGPFTDB
1,Updated: after first book arrived very damaged...,5.0,0593235657
2,I bought it for the bag on the front so it pai...,5.0,1782490671
3,Updated: after 1st arrived damaged the replace...,5.0,0593138228
4,I love this book! The patterns are lovely. I ...,5.0,0823098079
5,Missing the sketch pad. Even worse I realized ...,1.0,1631591290
6,Seriously one of only a few books they I have ...,4.0,1640210148
7,I love this book. I was not blessed with arti...,5.0,1784881953
8,I really wanted to like this book bc I have he...,3.0,1645671127
9,Every page has a crease running the entire len...,1.0,1780671067


In [70]:
check_meta = con.execute("""
    SELECT *
    FROM "../data/processed/meta_Books_processed.parquet"
    LIMIT 10
    """
)

display(check_meta.df())

,title,author,features,categories,average_rating,rating_number,price,parent_asin
0,Chaucer,Peter Ackroyd,,"Books, Literature & Fiction, History & Criticism",4.5,29,8.23,0701169850
1,Notes from a Kidwatcher,Yetta M. Goodman,Contains 23 selected articles by this influent...,"Books, Reference, Words, Language & Grammar",5.0,1,3.52,0435088688
2,Service: A Navy SEAL at War,Marcus Luttrell,"Marcus Luttrell, author of the #1 bestseller, ...","Books, Biographies & Memoirs, Leaders & Notabl...",4.7,3421,17.17,0316185361
3,Monstrous Stories #4: The Day the Mice Stood S...,,"Funny, light-hearted monster stories that are ...","Books, Children's Books, Science Fiction & Fan...",4.4,40,7.43,0545425573
4,Parker & Knight,Donald Wells,"From REMINGTON KANE, the author of The Taken! ...","Books, Mystery, Thriller & Suspense, Thrillers...",4.5,381,0.0,B00KFOP3RG
5,Writings from a Black Woman Living in the Land...,,Take a step into the modern perspective of a y...,"Books, Arts & Photography, History & Criticism",5.0,5,4.05,B09PHG4FQ8
6,Child Development: A Practitioner's Guide:2nd ...,,"Child Development, Second EditionDouglas Davies","Books, Parenting & Relationships, Parenting",5.0,2,10.68,B0086HQWC4
7,Make: Electronics: Learning Through Discovery,Charles Platt,"""This is teaching at its best!"", Hans Camenzin...","Books, Engineering & Transportation, Engineering",4.7,1366,13.43,1680450263
8,Reunion: The Children of Lauderdale Park,,"1940-Sadie, Jacob, Seth, and Hattie Lauderdale...","Books, Literature & Fiction, Genre Fiction",4.9,12,14.0,1694621731
9,Four Centuries of American Education,David Barton,"For four centuries, religion, morality, and kn...","Books, Education & Teaching, Schools & Teaching",4.8,133,6.99,1932225323


## Save Processed Data

In [ ]:
# Note, save as that fancy format to use with llms??